# 02 — Training Benchmark: Delta vs Lance

**Purpose:** Read each format back through Ray Data and feed it to Ray Train, measuring where the storage format separates the two pipelines. Covers stages **4 (inline preprocess), 5 (train), 6 (compare)** of [`README.md`](README.md).

**Prerequisite:** Run `01_create_benchmark_datasets.ipynb` for the target `size` tier first.

| Branch | Read path |
|--------|-----------|
| **Lance** | `read_lance` → image bytes inline, no extra hop |
| **Delta** | `read_databricks_tables` (metadata + `image_path`) → per-image Volumes GET to fetch JPEG bytes |

The Delta per-image GET hop is the mechanism the parent blueprint's failure-mode #1 describes. At batch 64 that's 64 concurrent file reads per step — the thing that starves the GPU. Lance reads bytes directly.

| Section | What it measures |
|---------|------------------|
| **1 — Data loading throughput** | Read + decode/resize samples/sec; full-column vs projected read |
| **2 — Training throughput** | Dummy-model (I/O ceiling) and real ResNet-50 runs; samples/sec, time-to-first-batch, batch-latency p50/p95/p99 |
| **3 — Compare** | Side-by-side, logged to MLflow |

---

### GPU sizing

Transfer-learning **ResNet-50** on an **A10 (24GB)**: ~600–900 img/s/GPU at 224px with AMP — light compute, so a single GPU is easy to *starve*. That is what surfaces a data-loading bottleneck. Start at **4 × A10** (`num_workers=4`) → ~2.4–3.6k img/s aggregate demand, the range where the Delta GET hop and shuffled random-access reads diverge from Lance. Bump `num_gpu_workers` to 8 for the 1m/10m tiers.

In [ ]:
# Install BEFORE any ray init.
# Version policy: pin exact for reproducible benchmark numbers; refresh to latest stable
# by hand when revisited (Ray latest stable at authoring: 2.56.1, 2026-07).
# pylance = the Lance Python package on PyPI (you `import lance`); left unpinned to track latest.
# pyarrow floored, not pinned — DBR ML preinstalls it and an exact pin risks a runtime conflict.
%pip install -qU "ray[data,train]==2.56.1" pylance "pyarrow>=16.0" torch torchvision Pillow numpy pandas "mlflow<3.0,>=2.17" "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("warehouse_id", "", "SQL Warehouse ID (blank = provision)")
dbutils.widgets.text("num_gpu_workers", "4", "GPU workers (A10)")
dbutils.widgets.text("batch_size", "64", "Batch size")
dbutils.widgets.text("num_epochs", "3", "Epochs (steady-state)")
dbutils.widgets.text("mlflow_experiment", "", "MLflow experiment (blank = default)")

size            = dbutils.widgets.get("size")
catalog         = dbutils.widgets.get("catalog")
schema          = dbutils.widgets.get("schema")
volume          = dbutils.widgets.get("volume")
NUM_GPU_WORKERS = int(dbutils.widgets.get("num_gpu_workers"))
BATCH_SIZE      = int(dbutils.widgets.get("batch_size"))
NUM_EPOCHS      = int(dbutils.widgets.get("num_epochs"))

# MUST match 01_create_benchmark_datasets.ipynb.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]
IMG_SIZE   = 224

base_vol     = f"/Volumes/{catalog}/{schema}/{volume}"
lance_path   = f"{base_vol}/synthetic_lance_{size}"
delta_table  = f"{catalog}.{schema}.synthetic_delta_{size}"
ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
username      = notebook_path.split("/")[2]
mlflow_exp    = dbutils.widgets.get("mlflow_experiment") or f"/Users/{username}/delta-vs-lance-benchmark"

print(f"Size tier   : {size}")
print(f"GPU workers : {NUM_GPU_WORKERS} x A10")
print(f"Lance       : {lance_path}")
print(f"Delta table : {delta_table}")
print(f"MLflow exp  : {mlflow_exp}")

In [ ]:
import os

os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_db_host  = os.environ["DATABRICKS_HOST"]
_db_token = os.environ["DATABRICKS_TOKEN"]

## SQL Warehouse — provision or reuse

`ray.data.read_databricks_tables` (the Delta branch) routes through a running SQL Warehouse. Same provision-or-reuse helper as `01`.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import State

w = WorkspaceClient()
WAREHOUSE_NAME = "ray-benchmark-warehouse"


def get_or_create_warehouse(warehouse_id="", name=WAREHOUSE_NAME,
                            cluster_size="Small", auto_stop_mins=10):
    if warehouse_id:
        return warehouse_id
    for wh in w.warehouses.list():
        if wh.name == name:
            if wh.state in (State.STOPPED, State.STOPPING):
                w.warehouses.start(wh.id).result()
            elif wh.state == State.STARTING:
                w.warehouses.get_and_wait(wh.id)
            print(f"Reusing warehouse '{name}' ({wh.id})")
            return wh.id
    created = w.warehouses.create(
        name=name, cluster_size=cluster_size, auto_stop_mins=auto_stop_mins,
        enable_serverless_compute=True, min_num_clusters=1, max_num_clusters=1,
    ).result()
    print(f"Created warehouse '{name}' ({created.id})")
    return created.id


warehouse_id = get_or_create_warehouse(dbutils.widgets.get("warehouse_id"))

In [ ]:
# GPU Ray cluster — PATH A. Set spark.task.resource.gpu.amount = "0" in the cluster
# Spark config so Ray (not Spark) controls GPU allocation. One A10 per worker node.
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(
    min_worker_nodes=NUM_GPU_WORKERS,
    max_worker_nodes=NUM_GPU_WORKERS,
    num_gpus_worker_node=1,
    num_cpus_worker_node=16,
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

total_gpus = ray.cluster_resources().get("GPU", 0)
print(f"Total GPUs  : {total_gpus:.0f}")
assert total_gpus >= NUM_GPU_WORKERS, "GPUs missing — check spark.task.resource.gpu.amount = '0'"

## Shared preprocessing + readers

`decode_resize` fuses JPEG-decode → resize → normalize into the read path (CPU actors, parallel to GPU training). The two readers differ exactly where the formats differ:

- **Lance** — `read_lance(columns=["image","category"])`, bytes inline.
- **Delta** — `read_databricks_tables(query=...)` returns `image_path` + `category`, then `read_image_from_path` issues the per-image Volumes GET. This is the extra hop the benchmark isolates.

Column projection (`image`/`image_path` + `category` only — skipping caption/embedding/metadata) is where Lance's blob isolation shows up: metadata columns are never touched.

In [ ]:
import numpy as np

CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


def decode_resize(batch, img_size, cat_to_idx):
    """Fused decode → resize → normalize. Expects batch['image'] as JPEG bytes."""
    import io
    from PIL import Image

    imgs, labels = [], []
    for jpeg, cat in zip(batch["image"], batch["category"]):
        img = Image.open(io.BytesIO(bytes(jpeg))).convert("RGB").resize((img_size, img_size))
        arr = (np.asarray(img, dtype=np.float32) / 255.0).transpose(2, 0, 1)
        imgs.append(arr)
        labels.append(cat_to_idx[cat if isinstance(cat, str) else cat.decode()])
    return {"image": np.asarray(imgs, dtype=np.float32),
            "label": np.asarray(labels, dtype=np.int64)}


def read_image_from_path(batch):
    """Delta branch: fetch JPEG bytes from each image_path (per-image Volumes GET)."""
    out = []
    for p in batch["image_path"]:
        with open(p, "rb") as f:
            out.append(f.read())
    return {"image": np.asarray(out, dtype=object), "category": batch["category"]}


def read_format(fmt, warehouse_id, catalog, schema, delta_table, lance_path):
    """Branch-specific reader producing {'image': <bytes>, 'category': <str>} batches."""
    import ray
    if fmt == "lance":
        return ray.data.read_lance(lance_path, columns=["image", "category"])
    ds = ray.data.read_databricks_tables(
        warehouse_id=warehouse_id, catalog=catalog, schema=schema,
        query=f"SELECT image_path, category FROM {delta_table}",
    )
    return ds.map_batches(read_image_from_path, batch_size=64)   # the per-image GET hop

## Section 1 — Data loading throughput

Full pass over each format, decode/resize fused in, no GPU. The Delta branch includes the per-image GET; Lance reads inline. (Projected read only — the metadata columns are dropped in the query/projection, so full-column read is measured by widening the Delta `SELECT` / Lance `columns` if desired.)

In [ ]:
import time


def loading_throughput(fmt, batch_size):
    ds = read_format(fmt, warehouse_id, catalog, schema, delta_table, lance_path).map_batches(
        decode_resize, fn_kwargs={"img_size": IMG_SIZE, "cat_to_idx": CAT_TO_IDX},
        batch_size=batch_size,
    )
    t0 = time.time()
    n = 0
    for b in ds.iter_batches(batch_size=batch_size, batch_format="numpy"):
        n += len(b["label"])
    dt = time.time() - t0
    return {"samples_per_sec": round(n / dt, 1), "total": n, "elapsed_s": round(dt, 2)}


loading_results = {}
for fmt in ["delta", "lance"]:
    loading_results[fmt] = loading_throughput(fmt, BATCH_SIZE)
    r = loading_results[fmt]
    print(f"  {fmt:8s}: {r['samples_per_sec']:>10,.1f} samples/s ({r['total']:,} in {r['elapsed_s']}s)")

## Section 2 — Training throughput

ResNet-50 (ImageNet-pretrained backbone, fresh head) trained with `TorchTrainer` + DDP across the A10 workers, shuffled read per epoch. Two runs per format: **dummy** (compute skipped → data-loading ceiling) and **real** (full step → I/O-vs-compute attribution).

In [ ]:
def train_fn_per_worker(config):
    import time
    import numpy as np
    import torch
    import torch.nn as nn
    import ray.train, ray.train.torch
    from torchvision.models import resnet50, ResNet50_Weights
    from mlflow.tracking import MlflowClient

    device   = ray.train.torch.get_device()
    rank     = ray.train.get_context().get_world_rank()
    dummy    = config["dummy"]
    n_epochs = config["num_epochs"]
    bs       = config["batch_size"]

    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, config["num_classes"])
    model = ray.train.torch.prepare_model(model.to(device))
    opt   = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    lossf = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()

    shard  = ray.train.get_dataset_shard("train")
    client = MlflowClient() if rank == 0 else None

    for epoch in range(n_epochs):
        model.train()
        t_epoch = time.time()
        n, ttfb, batch_ms = 0, None, []
        for batch in shard.iter_torch_batches(batch_size=bs, dtypes=torch.float32, device=device):
            t_b = time.time()
            imgs = batch["image"]
            labels = batch["label"].long()
            if not dummy:
                with torch.cuda.amp.autocast():
                    loss = lossf(model(imgs), labels)
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                _ = imgs.mean()
            if ttfb is None:
                ttfb = time.time() - t_epoch
            batch_ms.append((time.time() - t_b) * 1000)
            n += imgs.shape[0]

        dt = time.time() - t_epoch
        p = np.percentile(batch_ms, [50, 95, 99])
        metrics = {
            "samples_per_sec": n / dt, "epoch_wall_s": dt, "time_to_first_batch_s": ttfb,
            "batch_ms_p50": float(p[0]), "batch_ms_p95": float(p[1]), "batch_ms_p99": float(p[2]),
        }
        if client is not None:
            for k, v in metrics.items():
                client.log_metric(config["mlflow_run_id"], k, v, step=epoch)
        ray.train.report(metrics)

In [ ]:
import mlflow
import pandas as pd
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig

mlflow.set_experiment(mlflow_exp)
training_results = {}

for fmt in ["delta", "lance"]:
    for dummy in [True, False]:
        mode = "dummy" if dummy else "real"

        ds = read_format(fmt, warehouse_id, catalog, schema, delta_table, lance_path).map_batches(
            decode_resize, fn_kwargs={"img_size": IMG_SIZE, "cat_to_idx": CAT_TO_IDX},
            batch_size=BATCH_SIZE,
        ).random_shuffle()

        with mlflow.start_run(run_name=f"{fmt}_{mode}_{size}") as run:
            mlflow.log_params({"format": fmt, "mode": mode, "dataset_size": size,
                               "model": "resnet50", "batch_size": BATCH_SIZE,
                               "num_epochs": NUM_EPOCHS, "num_gpu_workers": NUM_GPU_WORKERS})
            trainer = TorchTrainer(
                train_loop_per_worker=train_fn_per_worker,
                train_loop_config={"dummy": dummy, "num_epochs": NUM_EPOCHS,
                                   "batch_size": BATCH_SIZE, "num_classes": len(CATEGORIES),
                                   "mlflow_run_id": run.info.run_id},
                scaling_config=ScalingConfig(num_workers=NUM_GPU_WORKERS, use_gpu=True),
                datasets={"train": ds},
                run_config=RunConfig(storage_path=ray_tmp_path),
            )
            result = trainer.fit()
            training_results[(fmt, mode)] = result.metrics
            mlflow.log_metrics({k: v for k, v in result.metrics.items()
                                if isinstance(v, (int, float))})
        print(f"  {fmt:8s} {mode:6s}: {result.metrics.get('samples_per_sec', 0):>10,.1f} samples/s")

## Section 3 — Compare

Expected divergence (per the README): the per-image GET hop, projected reads, and shuffled random-access training throughput — not raw sequential scan. If the *dummy* gap is large but the *real* gap shrinks, GPU compute is masking the format difference — the tell to scale the model down or the data tier up.

In [ ]:
rows = []
for fmt in ["delta", "lance"]:
    rows.append({
        "format": fmt,
        "load_sps":         loading_results[fmt]["samples_per_sec"],
        "train_dummy_sps":  round(training_results[(fmt, "dummy")].get("samples_per_sec", 0), 1),
        "train_real_sps":   round(training_results[(fmt, "real")].get("samples_per_sec", 0), 1),
        "real_ttfb_s":      round(training_results[(fmt, "real")].get("time_to_first_batch_s", 0), 2),
        "real_p99_ms":      round(training_results[(fmt, "real")].get("batch_ms_p99", 0), 1),
    })
summary = pd.DataFrame(rows)
display(summary)

l = summary[summary.format == "lance"].iloc[0]
d = summary[summary.format == "delta"].iloc[0]
if d.train_real_sps > 0:
    print(f"\nLance vs Delta @ {size}:")
    print(f"  data loading : {l.load_sps / max(1, d.load_sps):.2f}x")
    print(f"  train (real) : {l.train_real_sps / d.train_real_sps:.2f}x")

## Deferred training metrics

Need node/GPU-level instrumentation rather than in-loop timing:

- **GPU utilization %** — `nvidia-smi` / DCGM sampled during the run (the direct data-starvation signal; samples/sec is the in-loop proxy used here).
- **Object-store spill events / disk IOPS** — Ray dashboard during shuffled reads.
- **Actor-pool utilization** and **CPU utilization on preprocess actors** — Ray dashboard timelines. For Delta, this includes the per-image GET actors, which is where its extra hop should show as IO-wait.